# 03 — Build your own

This notebook builds the memory-NLS solver from scratch, step by step, using only NumPy. It is slower than the GPU version in the other notebooks but exposes every part of the algorithm.

**You will write:**
1. The initial Gaussian field
2. The Strang split-step integrator
3. The auxiliary-field memory update
4. The diagnostic measurements (norm, peak, FWHM)

By the end you will understand exactly what the equation is doing. The pedagogical content matters more than the runtime, so we work on a small 2D lattice for clarity.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Step 1: build the initial state

The initial state is a complex Gaussian, normalized so that $\int |\Psi|^2 d^2x = 1$. The complex phase encodes any initial momentum the field has (we keep it real for simplicity).

In [ ]:
N = 64
L = 20.0
sigma_0 = 1.2
dx = L / N

x = (np.arange(N) - N/2) * dx
X, Y = np.meshgrid(x, x, indexing='ij')
r2 = X**2 + Y**2

# Real Gaussian envelope, normalized
envelope = np.exp(-r2 / (2 * sigma_0**2))
psi = envelope.astype(np.complex128)
norm_0 = np.sqrt(np.sum(np.abs(psi)**2) * dx**2)
psi /= norm_0

print(f'Initial peak density |psi|^2 = {np.max(np.abs(psi)**2):.4f}')
print(f'Initial norm sqrt(integral |psi|^2 d2x) = {np.sqrt(np.sum(np.abs(psi)**2) * dx**2):.6f}')

plt.imshow(np.abs(psi)**2, cmap='viridis', extent=[-L/2, L/2, -L/2, L/2])
plt.colorbar(label='|psi|^2')
plt.title('Initial density')
plt.show()

## Step 2: build the kinetic propagator

The free-particle Hamiltonian is $\hat{H}_{\text{kin}} = -\frac{1}{2}\nabla^2$ (we set $\hbar = m = 1$). In Fourier space, this is just $\frac{1}{2}k^2$. The full-step propagator $U_k = \exp(-i \hat{H}_{\text{kin}} dt)$ is therefore a pointwise multiplication in $k$-space. We precompute it once.

In [ ]:
dt = 0.0025

kvec = np.fft.fftfreq(N, d=dx) * 2 * np.pi
kx, ky = np.meshgrid(kvec, kvec, indexing='ij')
k2 = kx**2 + ky**2
H_kin = 0.5 * k2
U_k = np.exp(-1j * H_kin * dt)

print(f'Maximum |k| on the lattice: {np.sqrt(np.max(k2)):.2f}')
print(f'Phase rotation per step at max |k|: {np.max(H_kin) * dt:.2f} radians')

## Step 3: set up the memory

The memory has two modes: a fast mode ($\nu_1 = 10$, $\tau_1 = 0.1$) and a slow mode ($\nu_2 = 0.5$, $\tau_2 = 2$). Each is an auxiliary field $y_j(\mathbf{x})$ initialized to zero. They evolve by $\partial_t y_j = \nu_j (\rho - y_j)$, which has the exact discrete solution $y_j(t+dt) = e^{-\nu_j dt} y_j(t) + (1 - e^{-\nu_j dt}) \rho(t)$.

In [ ]:
nus = [10.0, 0.5]
lambdas = [0.3, 0.1]   # 2D paper coupling
decays = [np.exp(-nu * dt) for nu in nus]
ys = [np.zeros_like(psi.real) for _ in nus]

print(f'Memory modes: nu = {nus}, lambda = {lambdas}')
print(f'Decay factors per step: {decays}')
print(f'Effective time constants tau = 1/nu: {[1/nu for nu in nus]}')
print(f'Total memory coupling Sigma_lambda = {sum(lambdas):.2f}')

## Step 4: the Strang split-step time integration

Each time step has five substeps:

1. **Half potential step in real space**: $\Psi \to \exp(-i V_{\text{tot}} dt/2) \Psi$, where $V_{\text{tot}} = \Lambda \rho + V_{\text{mem}}$.
2. **Full kinetic step in k-space**: $\Psi \to \text{IFFT}[U_k \cdot \text{FFT}[\Psi]]$.
3. **Half potential step in real space** with the updated density.
4. **Exact OU update** of the auxiliary fields.
5. **Stochastic increment** (skipped in this conservative run).

The splitting error is $O(dt^2)$. We use $\Lambda = -8$ (the supercritical regime in 2D).

In [ ]:
Lambda = -8.0
n_steps = 2000

# Trajectory of the peak density and the L2 norm
history = {'t': [], 'peak': [], 'norm': []}

import time
t0 = time.time()

for step in range(n_steps):
    rho = np.abs(psi)**2
    V_mem = sum(lam * y for lam, y in zip(lambdas, ys))
    V_tot = Lambda * rho + V_mem
    psi = psi * np.exp(-0.5j * V_tot * dt)
    psi = np.fft.ifft2(U_k * np.fft.fft2(psi))
    rho = np.abs(psi)**2
    V_tot = Lambda * rho + V_mem
    psi = psi * np.exp(-0.5j * V_tot * dt)
    for i, (decay, _) in enumerate(zip(decays, nus)):
        ys[i] = decay * ys[i] + (1 - decay) * rho
    
    if step % 50 == 0:
        history['t'].append(step * dt)
        history['peak'].append(np.max(rho))
        history['norm'].append(np.sqrt(np.sum(rho) * dx**2))

print(f'Done. Wall time: {time.time() - t0:.1f}s on CPU.')
print(f'Final peak: {history["peak"][-1]:.4f}')
print(f'Norm drift: {abs(history["norm"][-1] - history["norm"][0]):.6f}')

## Step 5: visualize the trajectory and the final state

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].semilogy(history['t'], history['peak'])
axes[0].set_xlabel('time'); axes[0].set_ylabel('peak density (log)')
axes[0].set_title('Peak density vs. time')
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['t'], history['norm'])
axes[1].set_xlabel('time'); axes[1].set_ylabel('||psi||')
axes[1].set_title('L2 norm (should be conserved)')
axes[1].grid(True, alpha=0.3)

axes[2].imshow(np.abs(psi)**2, cmap='viridis', extent=[-L/2, L/2, -L/2, L/2])
axes[2].set_title('Final density')
plt.tight_layout()
plt.show()

## What you should see

The peak density rises sharply early in the trajectory (around $t \approx 2$), then declines by approximately three orders of magnitude. This is the 2D anti-collapse signature: the memory potential overshoots its equilibrium value during the collapse spike, releasing the field outward.

The L2 norm stays essentially constant — this is the conservative regime, where the equation is unitary (no $\Gamma$, no $\eta$). Any drift in the norm reflects only the splitting error of the Strang scheme.

The final density is much more spread out than the initial Gaussian, and exhibits the periodic crystalline structure documented in [`../results/02-spontaneous-crystallization.md`](../results/02-spontaneous-crystallization.md). At $N = 64$ the resolution is coarse, but the pattern is visible.

## What you have built

You have implemented:
- The complex Gaussian initial state with normalization (P1 — oscillation).
- The cubic Gross–Pitaevskii nonlinearity $\Lambda \rho \Psi$ (P2 instantaneous part).
- The multi-mode auxiliary-field memory potential (P2 across-time part).
- The Strang split-step time integration with FFT-based kinetic step.
- The norm conservation diagnostic.

What you have not implemented:
- Dissipation $-i\Gamma\Psi$.
- FDT-locked stochastic forcing $\eta$ (P3).
- Spinor structure with Pauli decomposition.
- Fractional dispersion.

These are present in the full solver in [`../implementation/physics/`](../implementation/physics/) and you can read that code to see how each is incorporated.